# Semana 01 · Tracking MLOps con MLflow en Databricks

En esta práctica vas a entrenar modelos **muy sencillos** para centrarte en el ciclo MLOps: datos, parámetros, métricas, artefactos, comparación y una decisión reproducible. El objetivo no es ganar una competición de machine learning, sino dejar evidencia clara en MLflow.


## Mapa y criterios de terminación

Al acabar debes poder enseñar:

1. un experimento de MLflow con varios runs comparables;
2. tags que identifiquen tu alias, lote y propósito docente;
3. parámetros y métricas de validación/test;
4. un modelo ganador guardado como artefacto;
5. un pequeño smoke test de inferencia.


## 0. Dependencias

En Databricks, ejecuta `%pip` al principio. Si el entorno reinicia Python, continúa desde la celda de imports.


In [0]:
%pip install -q --upgrade "mlflow[databricks]" "scikit-learn" "pandas<3" "matplotlib"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python


## 1. Configuración e identidad del lote

Usa rutas de Databricks (`/dbfs/...`) para artefactos pequeños que quieras inspeccionar fuera de MLflow.


In [0]:
import json
import time
import uuid
#Sirve para trabajar con rutas y archivos de forma más limpia.
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split #para dividir el dataset en train y test
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42 #semilla para que las partes aleatorias del experimento puedan ser reproducibles
STUDENT_ALIAS = "id02"  # elijo un identificador anonimo. Mas adelante MLFlow va a guardar ese alias como una etiqueta para que puedas distinguir mis ejecuciones de las de los demas 
BATCH_ID = f"s01-{uuid.uuid4().hex[:8]}" #genero un identificador unico para esta tanda de ejecuciones
EXPERIMENT_NAME = "/Shared/semana01-tracking" #me he creado una carpeta shared porque me daba error

#ARTIFACT_DIR = Path("/dbfs/tmp/muiaap_s01_tracking")

ARTIFACT_DIR = Path("/tmp/muiaap_s01_tracking") #definde donde guardar temporalmente archivos que genere el experimento 
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True) #crea esa carpeta si todavia no existe 

mlflow.set_experiment(EXPERIMENT_NAME)
print({"batch_id": BATCH_ID, "alias": STUDENT_ALIAS, "artifact_dir": str(ARTIFACT_DIR)})

{'batch_id': 's01-13ed1f1a', 'alias': 'id02', 'artifact_dir': '/tmp/muiaap_s01_tracking'}


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


## 2. Datos pequeños y contrato mínimo

Usaremos `load_wine` de scikit-learn: no descarga datos, es rápido y suficiente para practicar tracking.


In [0]:
wine = load_wine(as_frame=True)
data = wine.frame.rename(columns={"target": "quality_class"})
FEATURES = list(wine.feature_names)
TARGET = "quality_class"

assert TARGET in data.columns
assert data[FEATURES].isna().sum().sum() == 0
assert data[TARGET].nunique() == 3

display(data.head())
print(data.shape, data[TARGET].value_counts().sort_index().to_dict())

alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,quality_class
14.23,1.71,2.43,15.6,127.0,2.8,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
13.2,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.4,1050.0,0
13.16,2.36,2.67,18.6,101.0,2.8,3.24,0.3,2.81,5.68,1.03,3.17,1185.0,0
14.37,1.95,2.5,16.8,113.0,3.85,3.49,0.24,2.18,7.8,0.86,3.45,1480.0,0
13.24,2.59,2.87,21.0,118.0,2.8,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


(178, 14) {0: 59, 1: 71, 2: 48}


## 3. Separar train/validación/test

La celda ya trae una partición sencilla para que la práctica sea ejecutable en Databricks.

**TODO 2:** revisa si el reparto 70/15/15 mantiene todas las clases en validación y test. Decide si cambiarías el porcentaje de test en un proyecto real, porque la evaluación debe ser comparable y suficientemente estable. Inspecciona `train_test_split` y el parámetro `stratify`. Verifica con los `assert` y los conteos por clase.

en el proyecto real yo cambiaria el reparto a 60/20/20, porque como el dataset es pequeño (178 filas), un test del 15% deja muy pocos ejemplos por clase y cada fallo cambia mucho la nota, asi que con un 20% la evaluación es más fiable


In [0]:
X = data[FEATURES]
y = data[TARGET]

# Partición base 70/15/15. TODO 2: revisa los tamaños y conteos de clase antes de aceptar este split.

X_train, X_temp, y_train, y_temp = train_test_split(
    X, 
    y, 
    test_size=0.30, 
    random_state=SEED, 
    stratify=y #mantiene la proporcion de la variable objetivo
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, 
    y_temp, 
    test_size=0.50, 
    random_state=SEED, 
    stratify=y_temp
)

assert len(X_train) + len(X_valid) + len(X_test) == len(data)
print({"train": len(X_train), "valid": len(X_valid), "test": len(X_test)})
print("valid classes:", y_valid.value_counts().sort_index().to_dict())
print("test classes:", y_test.value_counts().sort_index().to_dict())


{'train': 124, 'valid': 27, 'test': 27}
valid classes: {0: 9, 1: 10, 2: 8}
test classes: {0: 9, 1: 11, 2: 7}


## 4. Candidatos simples

Mantén pocos candidatos para que el foco esté en MLflow. Un baseline, una regresión logística y un bosque pequeño son suficientes.

**TODO 3:** revisa los parámetros y decide si cambiarías uno. Importa porque los parámetros deben explicar diferencias entre runs. Inspecciona la API del estimador elegido y verifica que cada candidato tiene nombre y modelo.


In [0]:
CANDIDATES = [
    {
        "name": "baseline_most_frequent",
        "model": DummyClassifier(strategy="most_frequent"),
        "params": {"family": "baseline", "strategy": "most_frequent"},
    },
    #params es lo que luego se apunta en MLflow para cada run, como una ficha que dice "este run usó estos ajustes"

    #La regresion logistica tiene un parametro muy importante llamado c. controla cuanto se frena al modelo para que no memorice los datos de train. 

    #Scikit learn usa su valor por defecto (1.0). Mlflow guarda lo que pone en params. Si pobramos con (0.1) el resultado cambia. en MLflow ambos runs apareceran pero con distintos resultados auqnue tengan exactamente los mismos parametros. 
    {
        "name": "logistic_regression",
        "model": Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=500, random_state=SEED)),
        ]),
        "params": {"family": "linear","C":1.0, "max_iter": 500},
    },
    {
        "name": "random_forest_small",
        "model": RandomForestClassifier(n_estimators=60, max_depth=4, random_state=SEED),
        "params": {"family": "trees", "n_estimators": 60, "max_depth": 4}, #n:estimators=indica cuantos arboles hay, max_depth indica cuantas preguntas seguidas puede hacer cada arbol. 
    },
]

assert all("name" in c and "model" in c for c in CANDIDATES)
[c["name"] for c in CANDIDATES]

['baseline_most_frequent', 'logistic_regression', 'random_forest_small']

In [0]:
#Para comprobar la API del estimador:
#utilizo get_params() para ver que parametros tiene por defecto.
LogisticRegression().get_params()

{'C': 1.0,
 'class_weight': None,
 'dual': False,
 'fit_intercept': True,
 'intercept_scaling': 1,
 'l1_ratio': 0.0,
 'max_iter': 100,
 'n_jobs': None,
 'penalty': 'deprecated',
 'random_state': None,
 'solver': 'lbfgs',
 'tol': 0.0001,
 'verbose': 0,
 'warm_start': False}

In [0]:
#compruebo que cada candidato tiene nombre, modelo y ficha
assert all("name" in c and "model" in c and "params" in c for c in CANDIDATES)
print("✅ Todos los candidatos tienen nombre, modelo y params:", [c["name"] for c in CANDIDATES])

✅ Todos los candidatos tienen nombre, modelo y params: ['baseline_most_frequent', 'logistic_regression', 'random_forest_small']


In [0]:
#compruebo que no hay candidatos con el mismo nombre
nombres = [c["name"] for c in CANDIDATES]
assert len(nombres) == len(set(nombres))
print("✅ No hay nombres repetidos:", nombres)

✅ No hay nombres repetidos: ['baseline_most_frequent', 'logistic_regression', 'random_forest_small']


In [0]:
#compruebo que la ficha diga la verdad. para cada candidato se comprueban los valores escritos en params con los reales de get_params(): 
for c in CANDIDATES:
    reales = c["model"].get_params()
    for clave, valor in c["params"].items():
        if clave == "family":
            continue
        clave_real = clave if clave in reales else f"clf__{clave}"
        assert clave_real in reales
        assert reales[clave_real] == valor
print("✅ Los params de todos los candidatos coinciden con sus modelos reales")

✅ Los params de todos los candidatos coinciden con sus modelos reales


## 5. Registrar runs comparables

Cada run debe contar la misma historia: identidad, parámetros, métrica de validación y artefactos mínimos.

**TODO 4:** completa o revisa los tags. Importa porque después filtrarás sólo tus runs. Inspecciona `mlflow.set_tags`, `mlflow.log_params`, `mlflow.log_metrics` y `mlflow.sklearn.log_model`. Verifica que `run_infos` contiene un `model_uri` por candidato.


In [0]:
run_infos = []

for candidate in CANDIDATES:
    with mlflow.start_run(run_name=f"{BATCH_ID}-{candidate['name']}") as run:
        #tenemos que revisar los tags porque ahora mismo solo identifican quien y cuando (curso, semana, alumna) pero ninguno dice que modelo es cada run. 
        #esto es un poco como un manifest.json
        mlflow.set_tags({
            "course": "operacion-modelos",
            "week": "01",
            "student.alias": STUDENT_ALIAS,
            "batch_id": BATCH_ID,
            "purpose": "guided-mlflow-tracking",
            #estos dos los añado yo
            "model_family": candidate["params"]["family"],
            "split": "60/20/20",
        })
        mlflow.log_params(candidate["params"])
        mlflow.log_param("feature_count", len(FEATURES))
        mlflow.log_param("train_rows", len(X_train))
        mlflow.log_param("validation_rows", len(X_valid))

        start = time.perf_counter()
        model = candidate["model"]
        model.fit(X_train, y_train)
        valid_pred = model.predict(X_valid)
        latency_ms = (time.perf_counter() - start) * 1000

        metrics = {
            "validation.accuracy": accuracy_score(y_valid, valid_pred),
            "validation.f1_macro": f1_score(y_valid, valid_pred, average="macro"),
            "train_latency_ms": latency_ms,
        }
        mlflow.log_metrics(metrics)

        report = classification_report(y_valid, valid_pred, output_dict=True) #crea un informe con precision, recall y f1 por clase. se guarda como json y se sube con mlflow.log_artifact
        report_path = ARTIFACT_DIR / f"{candidate['name']}_validation_report.json"
        report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
        mlflow.log_artifact(str(report_path), artifact_path="validation")
        model_info = mlflow.sklearn.log_model(
            model,
            name="model",
            input_example=X_train.head(3),
            serialization_format="cloudpickle",
        ) #he tenido que cambair esto porque me estaba dando error. 
        #Guardar el modelo = serializarlo (convertirlo en un archivo para poder cargarlo después). MLflow usa por defecto el formato "skops", que es más seguro pero no acepta los árboles del random forest (tipo sklearn.tree._tree.Tree), y daba error al guardarlo.Por eso uso "cloudpickle", que guarda el modelo tal cual sin revisar los tipos. 

        run_infos.append({
            "run_id": run.info.run_id,
            "name": candidate["name"],
            "model_uri": model_info.model_uri,
            **metrics,
        })

runs_df = pd.DataFrame(run_infos).sort_values("validation.f1_macro", ascending=False)
display(runs_df)
# Un run por candidato
assert len(runs_df) == len(CANDIDATES)
# Todos los candidatos están, con su nombre
assert set(runs_df["name"]) == {c["name"] for c in CANDIDATES}
# Cada run tiene un model_uri, y son todos distintos
assert runs_df["model_uri"].notna().all()
assert runs_df["model_uri"].nunique() == len(CANDIDATES)

2026/10/03 15:56:32 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-aed5b051-eef0-4e4d-9074-235c54dc2477/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/local_disk0/.ephemeral_nfs/envs/pythonEnv-aed5b051-eef0-4e4d-9074-235c54dc2477/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/local

run_id,name,model_uri,validation.accuracy,validation.f1_macro,train_latency_ms
e69f4b58515243d7a3905dd92d2690f8,random_forest_small,models:/m-be38d675601f470d867e088770c4a1fa,1.0,1.0,80.43839400011166
cbe4d22fc3404751b8c0e978306ab275,logistic_regression,models:/m-84c03ed4f2da447988913d903239ce5a,0.9629629629629629,0.9649122807017543,317.1312060001128
4eeb12daa18b435491a17a7d1b7e1e3b,baseline_most_frequent,models:/m-686e28a383b64122b9c6109423cdd15b,0.37037037037037035,0.1801801801801802,4.051713999842832


## 6. Buscar tus runs y elegir ganador sin mirar test

**TODO 5:** filtra por `student.alias` y `batch_id`. Importa porque en un workspace compartido hay runs de otras personas. Inspecciona `mlflow.search_runs` y verifica que no aparecen runs de otro lote.


In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
filter_string = f"tags.student.alias = '{STUDENT_ALIAS}' and tags.batch_id = '{BATCH_ID}'" 
searched = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=filter_string,
    order_by=["metrics.validation.f1_macro DESC"],
)

display(searched[["run_id", "tags.mlflow.runName", "metrics.validation.f1_macro", "metrics.validation.accuracy"]])
assert set(searched["tags.batch_id"]) == {BATCH_ID}

#elegir el mejor
best_row = searched.iloc[0]
BEST_RUN_ID = best_row["run_id"]
BEST_MODEL_URI = f"runs:/{BEST_RUN_ID}/model"
print({"best_run_id": BEST_RUN_ID, "best_model_uri": BEST_MODEL_URI})

run_id,tags.mlflow.runName,metrics.validation.f1_macro,metrics.validation.accuracy
e69f4b58515243d7a3905dd92d2690f8,s01-13ed1f1a-random_forest_small,1.0,1.0
cbe4d22fc3404751b8c0e978306ab275,s01-13ed1f1a-logistic_regression,0.9649122807017543,0.9629629629629629
4eeb12daa18b435491a17a7d1b7e1e3b,s01-13ed1f1a-baseline_most_frequent,0.1801801801801802,0.37037037037037035


{'best_run_id': 'e69f4b58515243d7a3905dd92d2690f8', 'best_model_uri': 'runs:/e69f4b58515243d7a3905dd92d2690f8/model'}


In [0]:
# ¿Qué variables existen en la sesión?
for nombre in ["CANDIDATES", "BATCH_ID", "run_infos", "runs_df", "searched", "BEST_RUN_ID", "BEST_MODEL_URI"]:
    print(f"{nombre}: {'✅ existe' if nombre in globals() else '❌ no existe'}")

if "run_infos" in globals():
    print("\nrun_infos tiene", len(run_infos), "elementos")

CANDIDATES: ✅ existe
BATCH_ID: ✅ existe
run_infos: ✅ existe
runs_df: ✅ existe
searched: ✅ existe
BEST_RUN_ID: ✅ existe
BEST_MODEL_URI: ✅ existe

run_infos tiene 3 elementos


In [0]:
print("BATCH_ID:", BATCH_ID)
print("run_infos tiene", len(run_infos), "elementos (deberían ser", len(CANDIDATES), ")")
display(runs_df[["name", "model_uri", "validation.f1_macro"]])

BATCH_ID: s01-13ed1f1a
run_infos tiene 3 elementos (deberían ser 3 )


name,model_uri,validation.f1_macro
random_forest_small,models:/m-be38d675601f470d867e088770c4a1fa,1.0
logistic_regression,models:/m-84c03ed4f2da447988913d903239ce5a,0.9649122807017543
baseline_most_frequent,models:/m-686e28a383b64122b9c6109423cdd15b,0.1801801801801802


## 7. Evaluar test una sola vez

Test se usa después de elegir. Esto evita optimizar contra la evidencia final.

**TODO 6:** registra métricas de test en el run ganador. Importa porque la decisión queda trazada en el mismo run. Inspecciona `mlflow.start_run(run_id=...)` y verifica que las métricas `test.*` aparecen en MLflow.


In [0]:
best_model = mlflow.sklearn.load_model(BEST_MODEL_URI)
test_pred = best_model.predict(X_test)
test_metrics = {
    "test.accuracy": accuracy_score(y_test, test_pred),
    "test.f1_macro": f1_score(y_test, test_pred, average="macro"),
}

with mlflow.start_run(run_id=BEST_RUN_ID):
    mlflow.log_metrics(test_metrics)
    mlflow.set_tags({"selection.stage": "tested", "selection.reason": "best_validation_f1_macro"})

print(test_metrics)

{'test.accuracy': 1.0, 'test.f1_macro': 1.0}


In [0]:
run_guardado = mlflow.get_run(BEST_RUN_ID)
for nombre, valor in test_metrics.items():
    assert nombre in run_guardado.data.metrics, f"Falta la métrica {nombre} en MLflow"
    assert abs(run_guardado.data.metrics[nombre] - valor) < 1e-9, f"{nombre} no coincide"
print("✅ Métricas de test en el run ganador:", {k: v for k, v in run_guardado.data.metrics.items() if k.startswith("test.")})
print("Tags de selección:", {k: v for k, v in run_guardado.data.tags.items() if k.startswith("selection.")})

#Para verificar, guardo las metricas de test en el mismo run del modelo ganador, para que en un solo sitio se vea con qué se entrenó, por qué se eligió y cómo le fue en el test. en este caso, el ganador es random forest. 

✅ Métricas de test en el run ganador: {'test.accuracy': 1.0, 'test.f1_macro': 1.0}
Tags de selección: {'selection.reason': 'best_validation_f1_macro', 'selection.stage': 'tested'}


## 8. Smoke test de inferencia

Comprueba que un consumidor puede cargar el modelo por URI y predecir pocas filas. No levantamos API local ni usamos CLI: eso llegará en semanas posteriores.


In [0]:
#este test simula a una persona que no ha entrenado el modelo y solo quiere usarlo. 

# Simulo un consumidor: carga el modelo desde cero usando solo su URI de MLflow
modelo_consumidor = mlflow.pyfunc.load_model(BEST_MODEL_URI)

sample = X_test.head(3)
predictions = best_model.predict(sample)
result = pd.DataFrame({"prediction": predictions}, index=sample.index)
display(pd.concat([sample.reset_index(drop=True), result.reset_index(drop=True)], axis=1))
assert len(predictions) == 3
assert set(predictions).issubset(set(y.unique()))

# El modelo cargado por URI predice lo mismo que el modelo en memoria
assert (predictions == best_model.predict(sample)).all()
print("✅ El modelo se carga por URI y predice correctamente")

alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,prediction
14.06,2.15,2.61,17.6,121.0,2.6,2.51,0.31,1.25,5.05,1.06,3.58,1295.0,0
14.38,3.59,2.28,16.0,102.0,3.25,3.17,0.27,2.19,4.9,1.04,3.44,1065.0,0
12.93,3.8,2.65,18.6,102.0,2.41,2.41,0.25,1.98,4.5,1.03,3.52,770.0,0


✅ El modelo se carga por URI y predice correctamente


In [0]:
ficha = {
    "EXPERIMENT_NAME": EXPERIMENT_NAME,
    "BATCH_ID": BATCH_ID,
    "BEST_RUN_ID": BEST_RUN_ID,
    "Modelo ganador": best_row["tags.mlflow.runName"],
    "Métrica de validación (F1 macro)": round(best_row["metrics.validation.f1_macro"], 4),
    "Métrica de test (F1 macro)": round(test_metrics["test.f1_macro"], 4),
    "Métrica de test (accuracy)": round(test_metrics["test.accuracy"], 4),
}

for clave, valor in ficha.items():
    print(f"{clave}: {valor}")

EXPERIMENT_NAME: /Shared/semana01-tracking
BATCH_ID: s01-13ed1f1a
BEST_RUN_ID: e69f4b58515243d7a3905dd92d2690f8
Modelo ganador: s01-13ed1f1a-random_forest_small
Métrica de validación (F1 macro): 1.0
Métrica de test (F1 macro): 1.0
Métrica de test (accuracy): 1.0


## Entregable

Incluye en tu ficha: `EXPERIMENT_NAME`, `BATCH_ID`, `BEST_RUN_ID`, la métrica de validación usada para elegir, la métrica de test y una captura del smoke test.
